In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skyfield
from skyfield.api import load, wgs84, EarthSatellite
import astropy.coordinates
def angular_separation(lon1, lat1, lon2, lat2):
    sdlon = np.sin(lon2 - lon1)
    cdlon = np.cos(lon2 - lon1)
    slat1 = np.sin(lat1)
    slat2 = np.sin(lat2)
    clat1 = np.cos(lat1)
    clat2 = np.cos(lat2)

    num1 = clat2 * sdlon
    num2 = clat1 * slat2 - slat1 * clat2 * cdlon
    denominator = slat1 * slat2 + clat1 * clat2 * cdlon

    return np.arctan2(np.hypot(num1, num2), denominator)

def PredictedDoppler(ObjectName, line1, line2, station, year, month, day, hour, minute, second, Sps, tsp, ffs, timeoffset, frequencyinit,frequencysubtraction):
    import matplotlib.pyplot as plt
    import numpy as np
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import astropy.coordinates
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    days = beginningtimeofobservation - satellite.epoch
    #print('{:.3f} days away from epoch'.format(days))
    c = 299792458  # speed of light in m/s
    
    timebinsizeseconds = tsp[1]-tsp[0]
    HzPerBin = ffs[1]-ffs[0]
    
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    if station == 'Ke':
        difference = satellite - Katherine
    elif station == 'Hb':
        difference = satellite - mountpleasant
    elif station == 'Yg':
        difference = satellite - Yarragadee
    
    angle = 0
    timeintervalseconds = 1
    
    BinsPerMinute = 60/timebinsizeseconds
    BinsPerMinuteint = int(BinsPerMinute)
    
    frequency = frequencyinit*np.ones(observationlengthinminutes*BinsPerMinuteint)
    totaldopplershift = np.ones(observationlengthinminutes*BinsPerMinuteint)
    AngularDistanceStowed = np.ones(len(Sps))
    SatelliteLatitude = np.ones(len(Sps))
    DistanceFromObservationPoint = []
    Azimuth = []
    
    for i in range(observationlengthinminutes*BinsPerMinuteint):
        time1 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+timeoffset)
        time2 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+1+timeoffset)
        topocentricMP1 = difference.at(time1)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        topocentricMP2 = difference.at(time2)
        altMP2, azMP2, heightMP2 = topocentricMP2.altaz()
        displacementin1secondinmetres = (heightMP2.m - heightMP1.m)

        DistanceFromObservationPoint.append(heightMP1.km)
        Azimuth.append(azMP1.degrees)

        if i <= len(Sps)-1:
            lon1 = azMP1.degrees*np.pi/180
            lat1 = altMP1.degrees*np.pi/180
            lon2 = 1*np.pi/180
            lat2 = 88*np.pi/180
            AngularDistanceStowed[i] = angular_separation(lon1, lat1, lon2, lat2)*180/np.pi
            SatelliteLatitude[i] = lat1



        velocity = displacementin1secondinmetres/timeintervalseconds

        fdopplershift = (1*velocity*np.cos(angle)*frequencyinit)/(c - velocity)
        totaldopplershift[i] = -fdopplershift/HzPerBin #Hz per bin - Keep an eye on this value
        frequency[i] = frequencyinit + fdopplershift
        
    #self.DistanceFromObservationPoint = DistanceFromObservationPoint    
    #self.Azimuth = Azimuth 
    lenSps = len(Sps)
    tdslength = totaldopplershift[:lenSps]
    ExpectedFrequencies =  [(i * HzPerBin +int(frequencyinit-frequencysubtraction)) for i in tdslength]
    return(tdslength,ExpectedFrequencies)

def PredictedDopplerAboveHorizon(ObjectName, line1, line2, station, year, month, day, hour, minute, second, Sps, tsp, ffs, timeoffset, frequencyinit,frequencysubtraction,observationlengthinminutes):
    import matplotlib.pyplot as plt
    import numpy as np
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import astropy.coordinates
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    days = beginningtimeofobservation - satellite.epoch
    #print('{:.3f} days away from epoch'.format(days))
    c = 299792458  # speed of light in m/s
    
    timebinsizeseconds = tsp[1]-tsp[0]
    HzPerBin = ffs[1]-ffs[0]
    
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    if station == 'Ke':
        difference = satellite - Katherine
    elif station == 'Hb':
        difference = satellite - mountpleasant
    elif station == 'Yg':
        difference = satellite - Yarragadee
    
    angle = 0
    timeintervalseconds = 1
    
    BinsPerMinute = 60/timebinsizeseconds
    BinsPerMinuteint = int(BinsPerMinute)
    
    frequency = frequencyinit*np.ones(observationlengthinminutes*BinsPerMinuteint)
    totaldopplershift = np.ones(observationlengthinminutes*BinsPerMinuteint)
    AngularDistanceStowed = np.ones(len(Sps))
    SatelliteLatitude = np.ones(len(Sps))
    DistanceFromObservationPoint = []
    Azimuth = []
    Altitude = []
    
    TimeAboveHorizon = []
    FrequencyAboveHorizon = []
    
    for i in range(observationlengthinminutes*BinsPerMinuteint):
        time1 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+timeoffset)
        time2 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+1+timeoffset)
        topocentricMP1 = difference.at(time1)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        topocentricMP2 = difference.at(time2)
        altMP2, azMP2, heightMP2 = topocentricMP2.altaz()
        displacementin1secondinmetres = (heightMP2.m - heightMP1.m)
        
        

        DistanceFromObservationPoint.append(heightMP1.km)
        Azimuth.append(azMP1.degrees)
        Altitude.append(altMP1.degrees)

        if i <= len(Sps)-1:
            lon1 = azMP1.degrees*np.pi/180
            lat1 = altMP1.degrees*np.pi/180
            lon2 = 1*np.pi/180
            lat2 = 88*np.pi/180
            AngularDistanceStowed[i] = angular_separation(lon1, lat1, lon2, lat2)*180/np.pi
            SatelliteLatitude[i] = lat1



        velocity = displacementin1secondinmetres/timeintervalseconds

        fdopplershift = (1*velocity*np.cos(angle)*frequencyinit)/(c - velocity)
        
        totaldopplershift[i] = -fdopplershift/HzPerBin #Hz per bin - Keep an eye on this value
        frequency[i] = frequencyinit + fdopplershift
        
        
    #self.DistanceFromObservationPoint = DistanceFromObservationPoint    
    #self.Azimuth = Azimuth 
    lenSps = len(Sps)
    tdslength = totaldopplershift[:lenSps]
    ExpectedFrequencies =  [(i * HzPerBin +int(frequencyinit-frequencysubtraction)) for i in tdslength]
    print(Altitude)
    #print(ExpectedFrequencies)
    for i in range(len(tdslength)):
        if Altitude[i] > 0:############################################################################CheckThisValue
            FrequencyAboveHorizon.append(ExpectedFrequencies[i])
            TimeAboveHorizon.append(i+(tsp[1]-tsp[0])*0.5)
    return(FrequencyAboveHorizon,TimeAboveHorizon)

def AzimuthElevation(ObjectName, line1, line2, station, year, month, day, hour, minute, second, Sps, tsp, ffs, timeoffset, frequencyinit,frequencysubtraction):
    import matplotlib.pyplot as plt
    import numpy as np
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import astropy.coordinates
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    days = beginningtimeofobservation - satellite.epoch
    #print('{:.3f} days away from epoch'.format(days))
    c = 299792458  # speed of light in m/s
    
    timebinsizeseconds = tsp[1]-tsp[0]
    HzPerBin = ffs[1]-ffs[0]
    
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    if station == 'Ke':
        difference = satellite - Katherine
    elif station == 'Hb':
        difference = satellite - mountpleasant
    elif station == 'Yg':
        difference = satellite - Yarragadee
    
    angle = 0
    timeintervalseconds = 1
    
    BinsPerMinute = 60/timebinsizeseconds
    BinsPerMinuteint = int(BinsPerMinute)
    
    frequency = frequencyinit*np.ones(observationlengthinminutes*BinsPerMinuteint)
    totaldopplershift = np.ones(observationlengthinminutes*BinsPerMinuteint)
    AngularDistanceStowed = np.ones(len(Sps))
    SatelliteLatitude = np.ones(len(Sps))
    DistanceFromObservationPoint = []
    Azimuth = []
    Altitude = []
    
    
    for i in range(observationlengthinminutes*BinsPerMinuteint):
        time1 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+timeoffset)
        time2 = ts.utc(year, month, day, hour, minute+0, second+timebinsizeseconds*i+1+timeoffset)
        topocentricMP1 = difference.at(time1)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        topocentricMP2 = difference.at(time2)
        altMP2, azMP2, heightMP2 = topocentricMP2.altaz()
        displacementin1secondinmetres = (heightMP2.m - heightMP1.m)

        DistanceFromObservationPoint.append(heightMP1.km)
        Azimuth.append(azMP1.degrees)
        Altitude.append(altMP1.degrees)

        if i <= len(Sps)-1:
            lon1 = azMP1.degrees*np.pi/180
            lat1 = altMP1.degrees*np.pi/180
            lon2 = 1*np.pi/180
            lat2 = 88*np.pi/180
            AngularDistanceStowed[i] = angular_separation(lon1, lat1, lon2, lat2)*180/np.pi
            SatelliteLatitude[i] = lat1



        velocity = displacementin1secondinmetres/timeintervalseconds

        fdopplershift = (1*velocity*np.cos(angle)*frequencyinit)/(c - velocity)
        totaldopplershift[i] = -fdopplershift/HzPerBin #Hz per bin - Keep an eye on this value
        frequency[i] = frequencyinit + fdopplershift
        
    #self.DistanceFromObservationPoint = DistanceFromObservationPoint    
    #self.Azimuth = Azimuth 
    lenSps = len(Sps)
    tdslength = totaldopplershift[:lenSps]
    ExpectedFrequencies =  [(i * HzPerBin +int(frequencyinit-frequencysubtraction)) for i in tdslength]
    return(Azimuth,Altitude)

def Roller(Sps,tdslength):
    Rolled=Sps.copy()
    for i in range(len(Sps)):
        Rolled[i] = np.roll(Rolled[i],-int(tdslength[i]),axis=0)
    Nspec = len(Sps)
    Aspec = np.sum(Rolled, axis=0) / Nspec
    return(Rolled,Aspec)

def RowRemover(Sps,RowRemoverThreshold,Type,RowsToRemoveManual):
    numberOfRowsToRemove = 0
    AvgFreqBinsAx0 = Sps.mean(axis=0)
    AvgFreqBinsAx1 = Sps.mean(axis=1)
    RowsToRemove = []
    for i in range(len(AvgFreqBinsAx0)):
        if AvgFreqBinsAx0[i] > RowRemoverThreshold:
            print(i)
            RowsToRemove.append(i)
            numberOfRowsToRemove += 1
    print('RowsToRemove =', RowsToRemove)
    print('numberOfRowsToRemove =', numberOfRowsToRemove)
    if Type == "Manual":
        for i in range(len(RowsToRemoveManual)):
            ip = RowsToRemoveManual[i]
            for j in range(len(Sps.mean(axis=1))):
                 Sps[j][ip] = np.median(Sps)
    else:
        for i in range(len(RowsToRemove)):
            ip = RowsToRemove[i]
            for j in range(len(Sps.mean(axis=1))):
                 Sps[j][ip] = np.median(Sps)
    
    return(Sps)

def ManualRowRemover(Sps,RowsToRemoveManual):
    RowReplacer = [np.median(Sps)]*len(Sps)
    TransposedSps = np.transpose(Sps)
    for i in range(len(RowsToRemoveManual)):
        TransposedSps[RowsToRemoveManual[i]] = RowReplacer
    Sps1 = np.transpose(TransposedSps)
    return(Sps1)

#ExtractValuesfromData
def MaxValueExtractor(tsp,ffs,Sps):
    MaxAmpInEachTimeBin = np.ones(len(tsp))
    AmpValues = np.ones(len(tsp))
    for i in range(len(tsp)):
        MaxAmpInEachTimeBin[i] = Sps[i].argmax()
        AmpValues[i] = np.max(Sps[i])
    MaxYvaluesIndex = np.ones(len(tsp))
    for i in range(len(tsp)):
         MaxYvaluesIndex[i] = ffs[0]+(MaxAmpInEachTimeBin[i])*(ffs[1]-ffs[0])
    return(AmpValues,MaxAmpInEachTimeBin,MaxYvaluesIndex)

def IntensityThreshold(tsp,Sps,ThresholdValue,AmpValues,MaxYvaluesIndex,ExpectedFrequencies):    
    ThresholdedAmpValues = []
    ThresholdedTime = []
    ThresholdedYindex = []
    ThresholdedExpectedFrequencies = []
    ThresholdedDatabaseExpectedFrequencies = []
    for i in range(len(tsp)-1):
        if (i > 0) and (i < len(tsp)):
            if (AmpValues[i] > ThresholdValue) and (AmpValues[i-1] > ThresholdValue or AmpValues[i+1] > ThresholdValue):
                ThresholdedAmpValues.append(AmpValues[i])
                ThresholdedTime.append(tsp[i])
                ThresholdedYindex.append(MaxYvaluesIndex[i])
                ThresholdedExpectedFrequencies.append(ExpectedFrequencies[i])
            else:
                continue
        else:
            continue
    
    numberOfDataPointsRetained = len(ThresholdedExpectedFrequencies)
    totalNumberOfDataPoints = len(ExpectedFrequencies)
    fractionOfDataPoints = len(ThresholdedExpectedFrequencies)/len(ExpectedFrequencies)
    
    
    TotalNumberOfPointsAboveThreshold = 0
    
    for i in range(len(Sps)):
        for j in range(len(Sps[i])):
            if Sps[i][j] > ThresholdValue:
                TotalNumberOfPointsAboveThreshold += 1
    
    totalfractionofpoints = TotalNumberOfPointsAboveThreshold/(len(Sps)*len(Sps[0])) 
    
    print('TotalNumberofDataPoints =',len(tsp))
    print("TotalSpsNumberOfPointsAboveThreshold =", TotalNumberOfPointsAboveThreshold)
    print("TotalSpsFractionOfPointsAboveThreshold =",totalfractionofpoints)
    print()
    print('numberOfDataPointsRetained =',numberOfDataPointsRetained)
    print('totalNumberOfDataPoints =',totalNumberOfDataPoints)
    print('fractionOfDataPoints=',fractionOfDataPoints)
    
    return(ThresholdedAmpValues,ThresholdedTime,ThresholdedYindex,ThresholdedExpectedFrequencies)

def OldIntensityThreshold(tsp,Sps,ThresholdValue,AmpValues,MaxYvaluesIndex):    
    ThresholdedAmpValues = []
    ThresholdedTime = []
    ThresholdedYindex = []
    ThresholdedExpectedFrequencies = []
    ThresholdedDatabaseExpectedFrequencies = []
    indicies = []
    for i in range(len(tsp)-1):
        if (i > 0) and (i < len(tsp)):
            if (AmpValues[i] > ThresholdValue) and (AmpValues[i-1] > ThresholdValue or AmpValues[i+1] > ThresholdValue):
                ThresholdedAmpValues.append(AmpValues[i])
                ThresholdedTime.append(tsp[i])
                ThresholdedYindex.append(MaxYvaluesIndex[i])
                indicies.append([i])
            else:
                continue
        else:
            continue
    
    numberOfDataPointsRetained = len(ThresholdedTime)
    totalNumberOfDataPoints = len(tsp)
    fractionOfDataPoints = len(ThresholdedTime)/len(tsp)
    
    
    TotalNumberOfPointsAboveThreshold = 0
    
    for i in range(len(Sps)):
        for j in range(len(Sps[i])):
            if Sps[i][j] > ThresholdValue:
                TotalNumberOfPointsAboveThreshold += 1
    
    totalfractionofpoints = TotalNumberOfPointsAboveThreshold/(len(Sps)*len(Sps[0])) 
    
    print('TotalNumberofDataPoints =',len(tsp))
    print("TotalSpsNumberOfPointsAboveThreshold =", TotalNumberOfPointsAboveThreshold)
    print("TotalSpsFractionOfPointsAboveThreshold =",totalfractionofpoints)
    print()
    print('numberOfDataPointsRetained =',numberOfDataPointsRetained)
    print('totalNumberOfDataPoints =',totalNumberOfDataPoints)
    print('fractionOfDataPoints=',fractionOfDataPoints)
    
    return(ThresholdedAmpValues,ThresholdedTime,ThresholdedYindex,indicies)

def RMSECalculator1(ThresholdedExpectedFrequencies,ThresholdedYindex):
    errorsquaresum = 0
    n_value = len(ThresholdedExpectedFrequencies)
    if n_value == 0:
        print("No Frequencies to Calculate")
    for i in range(n_value):
        errorsquared = (ThresholdedExpectedFrequencies[i] - ThresholdedYindex[i])**2
        errorsquaresum += errorsquared
    RMSE = np.sqrt(errorsquaresum/n_value)
    print('RMSE',RMSE,'Hz')
    return(RMSE)

def RMSECalculator2(ThresholdedExpectedFrequencies,ThresholdedYindex):
    residualList = []
    errorsquaresum = 0
    n_value = len(ThresholdedExpectedFrequencies)
    if n_value == 0:
        print("No Frequencies to Calculate")
    for i in range(n_value):
        residualList.append(ThresholdedExpectedFrequencies[i] - ThresholdedYindex[i])
        errorsquared = (ThresholdedExpectedFrequencies[i] - ThresholdedYindex[i])**2
        errorsquaresum += errorsquared
    RMSE = np.sqrt(errorsquaresum/n_value)
    print('RMSE',RMSE,'Hz')
    return(RMSE,residualList)

def Naive5pointSmoothing(ThresholdedYindex):
    SmoothedYIndexValues = []
    SmoothedYIndexValues.append(ThresholdedYindex[0]),SmoothedYIndexValues.append(ThresholdedYindex[1])
    for i in range(len(ThresholdedYindex)-4):
        fivePointAverage = int(0.2*(ThresholdedYindex[i]+ThresholdedYindex[i+1]+ThresholdedYindex[i+2]+ThresholdedYindex[i+3]+ThresholdedYindex[i+4]))
        SmoothedYIndexValues.append(fivePointAverage)
    SmoothedYIndexValues.append(ThresholdedYindex[-2]),SmoothedYIndexValues.append(ThresholdedYindex[-1])
    return(SmoothedYIndexValues)

def AdvancedSmoothing(xUnsmoothed,yUnsmoothed,timestepsize):
    XSmoothed = xUnsmoothed
    YSmoothed = []
    YSmoothed.append(yUnsmoothed[0])
    if xUnsmoothed[0] + timestepsize == xUnsmoothed[1] and xUnsmoothed[1] + timestepsize == xUnsmoothed[2]:
        YSmoothed.append((yUnsmoothed[0]+yUnsmoothed[1]+yUnsmoothed[2])/3)
    else:
        YSmoothed.append(yUnsmoothed[1])
    for i in range(len(xUnsmoothed)-4):
        if xUnsmoothed[i] + timestepsize == xUnsmoothed[i+1] and xUnsmoothed[i+1] + timestepsize == xUnsmoothed[i+2] and xUnsmoothed[i+2] + timestepsize == xUnsmoothed[i+3] and xUnsmoothed[i+3] + timestepsize == xUnsmoothed[i+4]:
            fivePointAverage = (0.2*(yUnsmoothed[i]+yUnsmoothed[i+1]+yUnsmoothed[i+2]+yUnsmoothed[i+3]+yUnsmoothed[i+4]))
            YSmoothed.append(fivePointAverage)
        elif xUnsmoothed[i+1] + timestepsize == xUnsmoothed[i+2] and xUnsmoothed[i+2] + timestepsize == xUnsmoothed[i+3]:
            threePointAverage = ((yUnsmoothed[i+1]+yUnsmoothed[i+2]+yUnsmoothed[i+3])/3)
            YSmoothed.append(threePointAverage)
        else:
            YSmoothed.append(yUnsmoothed[i+2])
    if xUnsmoothed[-3] + timestepsize == xUnsmoothed[-2] and xUnsmoothed[-2] + timestepsize == xUnsmoothed[-1]:
        YSmoothed.append((yUnsmoothed[-3]+yUnsmoothed[-2]+yUnsmoothed[-1])/3)
    else:
        YSmoothed.append(yUnsmoothed[-2])
    YSmoothed.append(yUnsmoothed[-1])
    return(YSmoothed)

def TLEIterator(line1,line2,TermToBeIterated,TermLowerBound,TermUpperBound,StepSize):
    #TermToBeIterated Choose one of: BStar FirstDerivMeanMotion Inclination RightAscensionAscendingNode Eccentricity ArgumentOfPerigee MeanAnomaly MeanMotion
    TermValues = np.arange(TermLowerBound,TermUpperBound,StepSize)
    TermValuesList = list(TermValues)
    line1list = []
    line2list = []
    ################################################################################################################
    if TermToBeIterated == "BStar":
        for ik in range(len(TermValues)):
            BStarValuesSci="{:.4e}".format(TermValues[ik])
            l1=list(line1)
            l1[54], l1[55], l1[56], l1[57], l1[58], l1[59], l1[60] = BStarValuesSci[0],BStarValuesSci[2],BStarValuesSci[3],BStarValuesSci[4],BStarValuesSci[5],BStarValuesSci[7],BStarValuesSci[9]
            line1 = "".join(l1)
            line1list.append(line1)

    ################################################################################################################
    elif TermToBeIterated == "FirstDerivMeanMotion":
        for ik in range(len(TermValues)):
            FirstDerivMeanMotionValuesik = "{:.8f}".format(TermValues[ik])
            FirstDerivMeanMotionValueString = str(FirstDerivMeanMotionValuesik)
            l1=list(line1)

            if FirstDerivMeanMotionValues[ik] >=0:
                #l1[34],l1[35],l1[36],l1[37],l1[38],l1[39],l1[40],l1[41],l1[42],l1[43] = FirstDerivMeanMotionValueString[0],FirstDerivMeanMotionValueString[1],FirstDerivMeanMotionValueString[2],FirstDerivMeanMotionValueString[3],FirstDerivMeanMotionValueString[4],FirstDerivMeanMotionValueString[5],FirstDerivMeanMotionValueString[6],FirstDerivMeanMotionValueString[7],FirstDerivMeanMotionValueString[8],FirstDerivMeanMotionValueString[9]
                l1[34],l1[35],l1[36],l1[37],l1[38],l1[39],l1[40],l1[41],l1[42] = FirstDerivMeanMotionValueString[1],FirstDerivMeanMotionValueString[2],FirstDerivMeanMotionValueString[3],FirstDerivMeanMotionValueString[4],FirstDerivMeanMotionValueString[5],FirstDerivMeanMotionValueString[6],FirstDerivMeanMotionValueString[7],FirstDerivMeanMotionValueString[8],FirstDerivMeanMotionValueString[9]
            elif FirstDerivMeanMotionValues < 0:
                l1[32],l1[33],l1[34],l1[35],l1[36],l1[37],l1[38],l1[39],l1[40],l1[41],l1[42] = FirstDerivMeanMotionValueString[0],FirstDerivMeanMotionValueString[1],FirstDerivMeanMotionValueString[2],FirstDerivMeanMotionValueString[3],FirstDerivMeanMotionValueString[4],FirstDerivMeanMotionValueString[5],FirstDerivMeanMotionValueString[6],FirstDerivMeanMotionValueString[7],FirstDerivMeanMotionValueString[8],FirstDerivMeanMotionValueString[9],FirstDerivMeanMotionValueString[10]
            #print("l1",l1)
            line1 = "".join(l1)
            #print("line1",line1)
            line1list.append(line1)
    ################################################################################################################
    elif TermToBeIterated == "Inclination":
        for ik in range(len(TermValues)):
            InclinationValuesik = "{:.4f}".format(TermValues[ik])
            InclinationValueString = str(InclinationValuesik)
            l2=list(line2)

            if InclinationValues[ik] < 10:
                l2[8],l2[9],l2[10],l2[11],l2[12],l2[13],l2[14],l2[15] = ' ',' ',InclinationValueString[0],InclinationValueString[1],InclinationValueString[2],InclinationValueString[3],InclinationValueString[4],InclinationValueString[5]
            elif (InclinationValues[ik] >=10) and (InclinationValues[ik] < 100):
                l2[8],l2[9],l2[10],l2[11],l2[12],l2[13],l2[14],l2[15] = ' ',InclinationValueString[0],InclinationValueString[1],InclinationValueString[2],InclinationValueString[3],InclinationValueString[4],InclinationValueString[5],InclinationValueString[6]
            elif (InclinationValues[ik] >= 100):
                l2[8],l2[9],l2[10],l2[11],l2[12],l2[13],l2[14],l2[15] = InclinationValueString[0],InclinationValueString[1],InclinationValueString[2],InclinationValueString[3],InclinationValueString[4],InclinationValueString[5],InclinationValueString[6],InclinationValueString[7]
            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################ 
    elif TermToBeIterated == "RightAscensionAscendingNode":
        for ik in range(len(TermValues)):
            RightAscensionAscendingNodeValuesik = "{:.4f}".format(TermValues[ik])
            RightAscensionAscendingNodeValueString = str(RightAscensionAscendingNodeValuesik)
            l2=list(line2)

            if RightAscensionAscendingNodeValues[ik] < 10:
                l2[17],l2[18],l2[19],l2[20],l2[21],l2[22],l2[23],l2[24] = ' ',' ',RightAscensionAscendingNodeValueString[0],RightAscensionAscendingNodeValueString[1],RightAscensionAscendingNodeValueString[2],RightAscensionAscendingNodeValueString[3],RightAscensionAscendingNodeValueString[4],RightAscensionAscendingNodeValueString[5]
            elif (RightAscensionAscendingNodeValues[ik] >=10) and (RightAscensionAscendingNodeValues[ik] < 100):
                l2[17],l2[18],l2[19],l2[20],l2[21],l2[22],l2[23],l2[24] = ' ',RightAscensionAscendingNodeValueString[0],RightAscensionAscendingNodeValueString[1],RightAscensionAscendingNodeValueString[2],RightAscensionAscendingNodeValueString[3],RightAscensionAscendingNodeValueString[4],RightAscensionAscendingNodeValueString[5],RightAscensionAscendingNodeValueString[6]
            elif (RightAscensionAscendingNodeValues[ik] >= 100):
                l2[17],l2[18],l2[19],l2[20],l2[21],l2[22],l2[23],l2[24] = RightAscensionAscendingNodeValueString[0],RightAscensionAscendingNodeValueString[1],RightAscensionAscendingNodeValueString[2],RightAscensionAscendingNodeValueString[3],RightAscensionAscendingNodeValueString[4],RightAscensionAscendingNodeValueString[5],RightAscensionAscendingNodeValueString[6],RightAscensionAscendingNodeValueString[7]
            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################ 
    elif TermToBeIterated == "Eccentricity":
        for ik in range(len(TermValues)):
            EccentricityValuesik = "{:.7f}".format(TermValues[ik])
            EccentricityValueString = str(EccentricityValuesik)
            l2=list(line2)

            l2[26],l2[27],l2[28],l2[29],l2[30],l2[31],l2[32] = EccentricityValueString[2],EccentricityValueString[3],EccentricityValueString[4],EccentricityValueString[5],EccentricityValueString[6],EccentricityValueString[7],EccentricityValueString[8]

            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################
    elif TermToBeIterated == "ArgumentOfPerigee":
        for ik in range(len(TermValues)):
            ArgumentOfPerigeeValuesik = "{:.4f}".format(TermValues[ik])
            ArgumentOfPerigeeValueString = str(ArgumentOfPerigeeValuesik)
            l2=list(line2)

            if ArgumentOfPerigeeValues[ik] < 10:
                l2[34],l2[35],l2[36],l2[37],l2[38],l2[39],l2[40],l2[41] = ' ',' ',ArgumentOfPerigeeValueString[0],ArgumentOfPerigeeValueString[1],ArgumentOfPerigeeValueString[2],ArgumentOfPerigeeValueString[3],ArgumentOfPerigeeValueString[4],ArgumentOfPerigeeValueString[5]
            elif (ArgumentOfPerigeeValues[ik] >=10) and (ArgumentOfPerigeeValues[ik] < 100):
                l2[34],l2[35],l2[36],l2[37],l2[38],l2[39],l2[40],l2[41] = ' ',ArgumentOfPerigeeValueString[0],ArgumentOfPerigeeValueString[1],ArgumentOfPerigeeValueString[2],ArgumentOfPerigeeValueString[3],ArgumentOfPerigeeValueString[4],ArgumentOfPerigeeValueString[5],ArgumentOfPerigeeValueString[6]
            elif (ArgumentOfPerigeeValues[ik] >= 100):
                l2[34],l2[35],l2[36],l2[37],l2[38],l2[39],l2[40],l2[41] = ArgumentOfPerigeeValueString[0],ArgumentOfPerigeeValueString[1],ArgumentOfPerigeeValueString[2],ArgumentOfPerigeeValueString[3],ArgumentOfPerigeeValueString[4],ArgumentOfPerigeeValueString[5],ArgumentOfPerigeeValueString[6],ArgumentOfPerigeeValueString[7]
            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################
    elif TermToBeIterated == "MeanAnomaly":
        for ik in range(len(TermValues)):
            MeanAnomalyValuesik = "{:.4f}".format(TermValues[ik])
            MeanAnomalyValueString = str(MeanAnomalyValuesik)
            l2=list(line2)

            if MeanAnomalyValues[ik] < 10:
                l2[43],l2[44],l2[45],l2[46],l2[47],l2[48],l2[49],l2[50] = ' ',' ',MeanAnomalyValueString[0],MeanAnomalyValueString[1],MeanAnomalyValueString[2],MeanAnomalyValueString[3],MeanAnomalyValueString[4],MeanAnomalyValueString[5]
            elif (MeanAnomalyValues[ik] >=10) and (MeanAnomalyValues[ik] < 100):
                l2[43],l2[44],l2[45],l2[46],l2[47],l2[48],l2[49],l2[50] = ' ',MeanAnomalyValueString[0],MeanAnomalyValueString[1],MeanAnomalyValueString[2],MeanAnomalyValueString[3],MeanAnomalyValueString[4],MeanAnomalyValueString[5],MeanAnomalyValueString[6]
            elif (MeanAnomalyValues[ik] >= 100):
                l2[43],l2[44],l2[45],l2[46],l2[47],l2[48],l2[49],l2[50] = MeanAnomalyValueString[0],MeanAnomalyValueString[1],MeanAnomalyValueString[2],MeanAnomalyValueString[3],MeanAnomalyValueString[4],MeanAnomalyValueString[5],MeanAnomalyValueString[6],MeanAnomalyValueString[7]

            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################
    elif TermToBeIterated == "MeanMotion":
        for ik in range(len(TermValues)):
            MeanMotionValuesik = "{:.17f}".format(TermValues[ik])
            MeanMotionValueString = str(MeanMotionValuesik)
            l2=list(line2)

            if MeanMotionValues[ik] < 10:
                l2[52],l2[53],l2[54],l2[55],l2[56],l2[57],l2[58],l2[59],l2[60],l2[61],l2[62],l2[63],l2[64],l2[65],l2[66],l2[67],l2[68] = ' ',MeanMotionValueString[0],MeanMotionValueString[1],MeanMotionValueString[2],MeanMotionValueString[3],MeanMotionValueString[4],MeanMotionValueString[5],MeanMotionValueString[6],MeanMotionValueString[7],MeanMotionValueString[8],MeanMotionValueString[9],MeanMotionValueString[10],MeanMotionValueString[11],MeanMotionValueString[12],MeanMotionValueString[13],MeanMotionValueString[14],MeanMotionValueString[15]
            elif (MeanMotionValues[ik] >=10) and (MeanMotionValues[ik] < 100):
                l2[52],l2[53],l2[54],l2[55],l2[56],l2[57],l2[58],l2[59],l2[60],l2[61],l2[62],l2[63],l2[64],l2[65],l2[66],l2[67],l2[68] = MeanMotionValueString[0],MeanMotionValueString[1],MeanMotionValueString[2],MeanMotionValueString[3],MeanMotionValueString[4],MeanMotionValueString[5],MeanMotionValueString[6],MeanMotionValueString[7],MeanMotionValueString[8],MeanMotionValueString[9],MeanMotionValueString[10],MeanMotionValueString[11],MeanMotionValueString[12],MeanMotionValueString[13],MeanMotionValueString[14],MeanMotionValueString[15],MeanMotionValueString[16]
            elif (MeanMotionValues[ik] >= 100):
                print("There are more than 100 orbits a day - ERROR")
            #print("l1",l1)
            line2 = "".join(l2)
            #print("line1",line1)
            line2list.append(line2)            
            print(line2list)
    ################################################################################################################

    if len(line1list) == 0:
        line1list.append(line1)
        for i in range(len(line2list)-1):
            line1list.append(line1)
    if len(line2list) == 0:
        line2list.append(line2)
        for i in range(len(line1list)-1):
            line2list.append(line1)
    return(line1list,line2list)
def skymaker(List,Difference):
    OutputList = []
    for i in range(len(List)):
        OutputList.append(List[i]+Difference)
    return(OutputList)
def AltitudeAzimuthDiagram(ObjectName, line1, line2, location, year, month, day, hour, minute, second, ObservationLengthInMinutes, LowestVisibleAltitude):
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import numpy as np
    from datetime import datetime
    from pytz import timezone
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    
    # List of all the locations of the telescopes
    mountpleasant = wgs84.latlon(-42.8055, 147.438)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    difference = satellite - mountpleasant
    
    #Sets the location
    if location == 'Hb':
        difference = satellite - mountpleasant
    elif location == 'Yg':
        difference = satellite - Yarragadee
    elif location == 'Ke':
        difference = satellite - Katherine
    elif location == 'Cd':
        difference = satellite - Ceduna
    elif location == 'NZ':
        difference = satellite - newZealand
        
        
        #create the observation lists from the TLE
    altitudeList = []
    azimuthList = []
    DistanceList = []
    visibletime = []
    altDifference = 0
    azDifference = 0
    altTime = 1 #second
    azTime = 1 #second
    
#    MaxAltSpeed = 0
#    MaxAzSpeed = 0
    
    LengthofObservation = ObservationLengthInMinutes*60 #(time in minutes)
    for i in range(LengthofObservation):
        #time and location now
        observationtime = ts.utc(year, month, day, hour, minute, second+i)
        topocentricHb = difference.at(observationtime)
        altHb, azHb, heightHb = topocentricHb.altaz()
        individualalt = [altHb.degrees]
        individualaz = [azHb.degrees]
        #time and location 1 second later
        observationtimeplus1 = ts.utc(year, month, day, hour, minute, second+i+1)
        topocentricHb1 = difference.at(observationtimeplus1)
        altHb1, azHb1, heightHb1 = topocentricHb1.altaz()
        individualalt1 = [altHb1.degrees]
        individualaz1 = [azHb1.degrees]
        individialHeight = [heightHb1.km]
        
        if altHb.degrees > LowestVisibleAltitude:
            altitudeList = np.append(altitudeList, individualalt, axis = 0)
            azimuthList = np.append(azimuthList, individualaz, axis = 0)
            DistanceList = np.append(DistanceList, individialHeight, axis = 0)
            visibletime.append(str(observationtime.utc_strftime()))
            
            #Finding the Maximum Altitude Speed
            if altDifference < abs(altHb.degrees-altHb1.degrees):
                altDifference = abs(altHb.degrees-altHb1.degrees)
                MaxAltSpeed = altDifference/altTime
            
            #Finding the Maximum Azimuthal Speed
            if azDifference < abs(azHb.degrees-azHb1.degrees):
                azDifference = abs(azHb.degrees-azHb1.degrees)
                MaxAzSpeed = azDifference/azTime
            
            
            
    AltitudeAzimuth = np.column_stack((altitudeList, azimuthList))

    azimuthRing = [i*np.pi/180 for i in range(361)]
    altitudeRing = 5*np.ones(361)
    
    #Create plot
    plt.style.use('dark_background')#, color = "lime")
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    plt.style.use('dark_background')
    ax = plt.subplot(polar=True)

    #horizon ring at five degrees
    ax.plot(azimuthRing, altitudeRing, color ="lime", linewidth=1.0)
    ax.set_theta_zero_location("N") #Puts zero at the top
    ax.set_theta_direction(-1) # Makes the graph go clockwise

    ax.set_rlim(90, -45, 1)
    ax.set_rmax(2)
    ax.set_rticks([10, 20, 30, 40, 50, 60, 70, 80, 90])#  # Radial ticks
    ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
    ax.grid(True)

    ax.tick_params(axis='x', colors='lime') #tick colours
    ax.tick_params(axis='y', colors='lime')

    ax.spines['polar'].set_color('lime')
    ax.spines['polar'].set_linewidth(1)

    ax.xaxis.grid(True,color='lime',linestyle='-') #grid colours
    ax.yaxis.grid(True,color='lime',linestyle='-')

    titlestring = 'Object: '+ObjectName+ '\n'+ 'Location: '+location+'\n'+'Rise Time: '+str(visibletime[0])+'\n'+'Set  Time: '+str(visibletime[-1])

    ax.set_title(titlestring, va='bottom', color = "white")

    x    = np.linspace(0,1, 100)
    y    = np.linspace(0,1, 100)
    cols = np.linspace(0,1,len(x))

    points = np.array([azimuthList*np.pi/180,altitudeList]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    lc = LineCollection(segments, cmap='Oranges_r') #YlGn
    lc.set_array(cols)
    lc.set_linewidth(5.5)
    line = ax.add_collection(lc)
    plt.show()
    
    largestAlt = 0
    largestAz = 0
    
    #ZenithTime = 0
    #ZenithHeight = 0

    #finds the point that the altitude is highest
    for i in range(len(altitudeList)-1):
        if altitudeList[i+1]>altitudeList[i]:
            largestAlt=altitudeList[i+1]
            largestAz=azimuthList[i+1]
            ZenithTime = visibletime[i+1]
            ZenithHeight = DistanceList[i+1]
        finalAlt = altitudeList[i]
        finalAz = azimuthList[i]
        finalHeight = DistanceList[i]
        #ZenithTime = visibletime[i]

    print('Satellite view begins: Altitude = ', altitudeList[0], 'degrees, Azimuth = ', azimuthList[0], 'degrees')
    print('Satellite zenith when: Altitude = ', largestAlt, 'degrees, Azimuth = ', largestAz, 'degrees')
    print('Satellite view ending: Altitude = ', finalAlt, 'degrees, Azimuth = ', finalAz, 'degrees')
    print('Satellite Rising Distance =',DistanceList[0],' km Satellite Zenith Distance =',ZenithHeight,' km Satellite Setting Distance =',finalHeight, 'km')
    print('Maximum Altitudinal Speed:', MaxAltSpeed, 'degrees per second')
    print('Maximum Azimuthal Speed:', MaxAzSpeed, 'degrees per second')
    print("Zenith Time:",ZenithTime)
    plt.style.use('default')
    
def UpperBoundSpectrumRemoved(Sps,UpperBoundThreshold):
    meanSps = np.mean(Sps)
    for i in range(len(Sps)):
        if Sps.mean(axis=1)[i] > UpperBoundThreshold:
            #print(i)
            for j in range(len(Sps[0])):
                Sps[i] = meanSps*np.ones(len(Sps[0]))
    return(Sps)

def LocationFinderAtTime(ObjectName, line1Unaltered, line2Unaltered, line1Altered, line2Altered, station, year, month, day, hour, minute, second):
    import matplotlib.pyplot as plt
    import numpy as np
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import astropy.coordinates
    ts = load.timescale()
    satellite1 = EarthSatellite(line1Unaltered, line2Unaltered, ObjectName, ts)
    satellite2 = EarthSatellite(line1Altered, line2Altered, ObjectName, ts)
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    
    zenithtime = ts.utc(year, month, day, hour, minute, second)
    
    days = zenithtime - satellite1.epoch
    c = 299792458  # speed of light in m/s
    if station == 'Ke':
        difference1 = satellite1 - Katherine
        difference2 = satellite2 - Katherine
    elif station == 'Hb':
        difference1 = satellite1 - mountpleasant
        difference2 = satellite2 - mountpleasant
    elif station == 'Yg':
        difference1 = satellite1 - Yarragadee
        difference2 = satellite2 - Yarragadee
        
    angle = 0
    timeintervalseconds = 1
    
        
    topocentricUnaltered = difference1.at(zenithtime)
    altUnaltered, azUnaltered, heightUnaltered = topocentricUnaltered.altaz()
    
    topocentricAltered = difference2.at(zenithtime)
    altAltered, azAltered, heightAltered = topocentricAltered.altaz()
    
    ChangeInPosition = difference2.at(zenithtime) - difference1.at(zenithtime)
    #altChange,azChange,heightChange = ChangeInPosition.altaz()
    heightChange = ChangeInPosition.distance()
    print("Original TLE Alt =",altUnaltered)
    print("Altered TLE Alt =",altAltered)
    #print("Change in Alt =",altChange)
    
    print("Original TLE Az =",azUnaltered)
    print("Altered TLE Az =",azAltered)
    #print("Change in Az =",azChange)
    
    print("Original TLE Displacement =",heightUnaltered)
    print("Altered TLE Displacement =",heightAltered)
    print("Change in Displacement =",'{:.8f} m'.format(heightChange.m))

def SpeedFinderAtTime(ObjectName, line1, line2, station, year, month, day, hour, minute, second):
    import matplotlib.pyplot as plt
    import numpy as np
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import astropy.coordinates
    ts = load.timescale()
    satellite1 = EarthSatellite(line1, line2, ObjectName, ts)
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    
    time = ts.utc(year, month, day, hour, minute, second)
    timePlus1 = ts.utc(year, month, day, hour, minute, second+1)
    timePlus10 = ts.utc(year, month, day, hour, minute, second+10)
    timePlus60 = ts.utc(year, month, day, hour, minute+1, second)
    
    days = time - satellite1.epoch
    c = 299792458  # speed of light in m/s
    if station == 'Ke':
        difference1 = satellite1 - Katherine        
    elif station == 'Hb':
        difference1 = satellite1 - mountpleasant
    elif station == 'Yg':
        difference1 = satellite1 - Yarragadee
                
    angle = 0
    timeintervalseconds = 1
    
    topocentric0 = difference1.at(time)
    alt0, az0, height0 = topocentric0.altaz()
    
    topocentric1 = difference1.at(timePlus1)
    alt1, az1, height1 = topocentric1.altaz()
    
    topocentric10 = difference1.at(timePlus10)
    alt10, az10, height10 = topocentric10.altaz()
    
    topocentric60 = difference1.at(timePlus60)
    alt60, az60, height60 = topocentric60.altaz()
    
    
    
    
def VelocityCalculator(DopplerShiftInHz,Frequencyinit):
    c = 299792458
    v = c*(DopplerShiftInHz/(DopplerShiftInHz+Frequencyinit))
    return(v)

#AntennaLocationData
mountpleasant = wgs84.latlon(-42.805, 147.439)
newZealand = wgs84.latlon(-45.03861, 170.0958)
Yarragadee = wgs84.latlon(-29.04567, 115.34876)
Katherine = wgs84.latlon(-14.375464, 132.152373)
Ceduna = wgs84.latlon(-31.86883, 133.809799)

def MatrixSubtractor(Matrix,Subtraction):
    NewMatrix = []
    addTerm = 0
    for i in range(len(Matrix)):
        addTerm = Matrix[i] - Subtraction
        NewMatrix.append(addTerm)
    return(NewMatrix)
def round_to_1(x):
    return round(x, -int(floor(log10(abs(x)))))
def roundup(x):
    return int(math.ceil(x / 10.0)) * 10
def rounddown(x):
    return int(math.floor(x / 10.0))
import math
def rounddown_r(x):
    return int(math.floor(x / rbinnumber))
def rounddown_a(x):
    return int(math.floor(x*(abinnumber-1)/360))
def roundup_r(x):
    return int(math.ceil(x * (rbinnumber-1)/90))
def roundup_a(x):
    return int(math.ceil(x*(abinnumber-1)/360))

def AltitudeAzimuthDiagramMultiple(ObjectName, line1, line2, location, year, month, day, hour, minute, second, ObservationLengthInMinutes, LowestVisibleAltitude):
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import numpy as np
    from datetime import datetime
    from pytz import timezone
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    
    # List of all the locations of the telescopes
    mountpleasant = wgs84.latlon(-42.8055, 147.438)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    difference = satellite - mountpleasant
    
    #Sets the location
    if location == 'Hb':
        difference = satellite - mountpleasant
    elif location == 'Yg':
        difference = satellite - Yarragadee
    elif location == 'Ke':
        difference = satellite - Katherine
    elif location == 'Cd':
        difference = satellite - Ceduna
    elif location == 'NZ':
        difference = satellite - newZealand
        
        
        #create the observation lists from the TLE
    altitudeList = []
    azimuthList = []
    DistanceList = []
    visibletime = []
    altDifference = 0
    azDifference = 0
    altTime = 1 #second
    azTime = 1 #second
    
#    MaxAltSpeed = 0
#    MaxAzSpeed = 0
    
    LengthofObservation = ObservationLengthInMinutes*60 #(time in minutes)
    for i in range(LengthofObservation):
        #time and location now
        observationtime = ts.utc(year, month, day, hour, minute, second+i)
        topocentricHb = difference.at(observationtime)
        altHb, azHb, heightHb = topocentricHb.altaz()
        individualalt = [altHb.degrees]
        individualaz = [azHb.degrees]
        #time and location 1 second later
        observationtimeplus1 = ts.utc(year, month, day, hour, minute, second+i+1)
        topocentricHb1 = difference.at(observationtimeplus1)
        altHb1, azHb1, heightHb1 = topocentricHb1.altaz()
        individualalt1 = [altHb1.degrees]
        individualaz1 = [azHb1.degrees]
        individialHeight = [heightHb1.km]
        
        if altHb.degrees > LowestVisibleAltitude:
            altitudeList = np.append(altitudeList, individualalt, axis = 0)
            azimuthList = np.append(azimuthList, individualaz, axis = 0)
            DistanceList = np.append(DistanceList, individialHeight, axis = 0)
            visibletime.append(str(observationtime.utc_strftime()))
            
            #Finding the Maximum Altitude Speed
            if altDifference < abs(altHb.degrees-altHb1.degrees):
                altDifference = abs(altHb.degrees-altHb1.degrees)
                MaxAltSpeed = altDifference/altTime
            
            #Finding the Maximum Azimuthal Speed
            if azDifference < abs(azHb.degrees-azHb1.degrees):
                azDifference = abs(azHb.degrees-azHb1.degrees)
                MaxAzSpeed = azDifference/azTime
            
            
            
    AltitudeAzimuth = np.column_stack((altitudeList, azimuthList))

    azimuthRing = [i*np.pi/180 for i in range(361)]
    altitudeRing = 5*np.ones(361)
    
    #Create plot
    #plt.style.use('dark_background')#, color = "lime")
    #fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    #plt.style.use('dark_background')
    #ax = plt.subplot(polar=True)

    #horizon ring at five degrees
    #ax.plot(azimuthRing, altitudeRing, color ="lime", linewidth=1.0)
    #ax.set_theta_zero_location("N") #Puts zero at the top
    #ax.set_theta_direction(-1) # Makes the graph go clockwise

    #ax.set_rlim(90, -45, 1)
    #ax.set_rmax(2)
    #ax.set_rticks([10, 20, 30, 40, 50, 60, 70, 80, 90])#  # Radial ticks
    #ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
    #ax.grid(True)

    #ax.tick_params(axis='x', colors='lime') #tick colours
    #ax.tick_params(axis='y', colors='lime')

    #ax.spines['polar'].set_color('lime')
    #ax.spines['polar'].set_linewidth(1)

    #ax.xaxis.grid(True,color='lime',linestyle='-') #grid colours
    #ax.yaxis.grid(True,color='lime',linestyle='-')

    #titlestring = 'Object: '+ObjectName+ '\n'+ 'Location: '+location+'\n'+'Rise Time: '+str(visibletime[0])+'\n'+'Set  Time: '+str(visibletime[-1])

    #ax.set_title(titlestring, va='bottom', color = "white")

    x    = np.linspace(0,1, 100)
    y    = np.linspace(0,1, 100)
    cols = np.linspace(0,1,len(x))

    points = np.array([azimuthList*np.pi/180,altitudeList]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    #lc = LineCollection(segments, cmap='Oranges_r') #YlGn
    #lc.set_array(cols)
    #lc.set_linewidth(5.5)
    #line = ax.add_collection(lc)
    #plt.show()
    
    largestAlt = 0
    largestAz = 0
    
    #ZenithTime = 0
    #ZenithHeight = 0

    #finds the point that the altitude is highest
    for i in range(len(altitudeList)-1):
        if altitudeList[i+1]>altitudeList[i]:
            largestAlt=altitudeList[i+1]
            largestAz=azimuthList[i+1]
            ZenithTime = visibletime[i+1]
            ZenithHeight = DistanceList[i+1]
        finalAlt = altitudeList[i]
        finalAz = azimuthList[i]
        finalHeight = DistanceList[i]
        #ZenithTime = visibletime[i]

    print('Satellite view begins: Altitude = ', altitudeList[0], 'degrees, Azimuth = ', azimuthList[0], 'degrees')
    print('Satellite zenith when: Altitude = ', largestAlt, 'degrees, Azimuth = ', largestAz, 'degrees')
    print('Satellite view ending: Altitude = ', finalAlt, 'degrees, Azimuth = ', finalAz, 'degrees')
    print('Satellite Rising Distance =',DistanceList[0],' km Satellite Zenith Distance =',ZenithHeight,' km Satellite Setting Distance =',finalHeight, 'km')
    print('Maximum Altitudinal Speed:', MaxAltSpeed, 'degrees per second')
    print('Maximum Azimuthal Speed:', MaxAzSpeed, 'degrees per second')
    print("Zenith Time:",ZenithTime)
    #plt.style.use('default')
    return(segments)
def tleEpochPrinter(line1,line2,year,month,day,hour,minute,second):
    from skyfield.api import EarthSatellite
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, 'AIM', ts)
    t = ts.utc(year,month,day,hour,minute,second)
    days = t - satellite.epoch
    print('{:.3f} days away from epoch'.format(days))
    
def roundup_r(x,rbinnumber):
    return int(math.ceil(x * (rbinnumber-1)/90))
def roundup_a(x,abinnumber):
    return int(math.ceil(x*(abinnumber-1)/360))

def SideLobesMap(azim,elev,SubtractedThresholdedAmpValues,rbinnumber,abinnumber):
    import matplotlib.pyplot as plt
    ListOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    MedianOfIntensitiesByAngleBinMatrix = [[[[0] for z in range(0)] for x in range((rbinnumber))] for y in range((abinnumber))]
    #print(MedianOfIntensitiesByAngleBinMatrix)
    rbins = np.linspace(0, 90, rbinnumber)
    abins = np.linspace(0, 360* np.pi/180, abinnumber)       

    for i in range(len(SubtractedThresholdedAmpValues)):
        x_value = roundup_a(azim[i],abinnumber)#; print(x_value,pol017hbPass1ThresholdedAzimuth[i])
        y_value = roundup_r(elev[i],rbinnumber)#; print(y_value,pol017hbPass1ThresholdedAltitude[i])
        ListOfIntensitiesByAngleBinMatrix[x_value][y_value].append(SubtractedThresholdedAmpValues[i])
    #ListOfIntensitiesByAngleBinMatrix
    for i in range(len(MedianOfIntensitiesByAngleBinMatrix)):
        for j in range(len(MedianOfIntensitiesByAngleBinMatrix[0])):
            if len(ListOfIntensitiesByAngleBinMatrix[i][j])>0:
                MedianOfIntensitiesByAngleBinMatrix[i][j].append(np.median(ListOfIntensitiesByAngleBinMatrix[i][j]))

    bin_selection_r = []
    bin_selection_a = []
    MedianValues1Dlist = []

    for i in range(len(MedianOfIntensitiesByAngleBinMatrix)):
        for j in range(len(MedianOfIntensitiesByAngleBinMatrix[0])):
            if len(MedianOfIntensitiesByAngleBinMatrix[i][j])>0:
                bin_selection_r.append(j*90/rbinnumber)
                bin_selection_a.append(i*360*np.pi/(180*abinnumber))
                MedianValues1Dlist.append(MedianOfIntensitiesByAngleBinMatrix[i][j][0])

    hist, _, _ = np.histogram2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=MedianValues1Dlist)#,cmin = 1)
    #hist, _, _ = plt.hist2d(bin_selection_a, bin_selection_r, bins=(abins, rbins),weights=MedianValues1Dlist,cmin = 1)
    A, R = np.meshgrid(abins, rbins)

    # plot
    fig, ax = plt.subplots(subplot_kw=dict(projection="polar"))#,cmin = 1
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    #import matplotlib as mpl
    #cmap1 = mpl.cm.get_cmap('magma').copy()  # viridis is the default colormap for imshow
    #cmap.set_bad(color='red')
    #cmap1.set_bad(color='white')

    #my_cmap = mpl.cm.get_cmap("magma").copy()
    #my_cmap.set_under('white',1)
    
    pc = ax.pcolormesh(A, R, hist.T, cmap='magma', vmin=0, vmax=15)#my_cmap)#cmap=cmap1)
    #print(hist.T)
    fig.colorbar(pc)
    ax.grid(True)
    ax.set_rlim(bottom=90, top=0)
    ax.tick_params(axis='y', colors='white')
    plt.show()
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
def RMSECalculator(TLEtime,TLEfrequency,DownlinkTime,DownlinkFrequency):
    errorsquaresum = 0
    n_value = len(DownlinkTime)
    for i in range(n_value):
         for j in range(len(TLEtime)):
                if TLEtime[j] == DownlinkTime[i]:
                    errorsquared = (DownlinkFrequency[i] - TLEfrequency[j])**2
                    errorsquaresum += errorsquared
    postsmoothingRMSE = float(np.sqrt(errorsquaresum/n_value))
    print('RMSE',postsmoothingRMSE,'Hz')
    return(postsmoothingRMSE)
def mediangradientfinder(listarray):
    import numpy as np
    listofgradients = []
    for i in range(int(len(listarray)/2-1)):
        listofgradients.append(listarray[i+1]-listarray[i])
    for i in range(int(len(listarray)/2-1)):
        listofgradients.append(listarray[(int(len(listarray)/2)-1)+i]-listarray[(int(len(listarray)/2)-1)+i+1])
    mediangradient = np.median(listofgradients)
    return(mediangradient,listofgradients)
def triangletemplatemaker(initialvalue,gradient,arraysize):
    nextterm = initialvalue
    template = []
    for i in range(int((arraysize)/2)):
        template.append(initialvalue+i*gradient)
    halfwayvalue = template[-1]
    for i in range(int((arraysize)/2)):
        template.append(halfwayvalue-i*gradient)
    return(template)
def correlator2(array1,array2):
    import scipy as sp
    from scipy import signal
    Correlated = sp.signal.correlate(array1, array2, mode="same", method='direct')
    return(Correlated)
def correlatorlags(array1,array2):
    import scipy as sp
    from scipy import signal
    Correlated = sp.signal.correlation_lags(array1, array2)#, mode='same')
    return(Correlated)
def correlator(array1,array2):
    import scipy as sp
    from scipy import signal
    Correlated = sp.signal.correlate(array1, array2, mode="same", method='direct')
    lags = signal.correlation_lags(array1.size, array2.size, mode="same")
    return(Correlated,lags)
def scalar(array,scalar,additive):
    newarray = []
    for i in range(len(array)):
        newarray.append((scalar*array[i])+additive)
    return(newarray)
def simplefivepointsmoother(xUnsmoothed,yUnsmoothed,timestepsize):
    XSmoothed = xUnsmoothed
    YSmoothed = []
    YSmoothed.append(yUnsmoothed[0])
    YSmoothed.append(yUnsmoothed[1])
    for i in range(len(xUnsmoothed)-4):
        fivePointAverage = (0.2*(yUnsmoothed[i]+yUnsmoothed[i+1]+yUnsmoothed[i+2]+yUnsmoothed[i+3]+yUnsmoothed[i+4]))
        YSmoothed.append(fivePointAverage)
    YSmoothed.append(yUnsmoothed[-2])
    YSmoothed.append(yUnsmoothed[-1])
    return(XSmoothed,YSmoothed)
def fivepointsmoother(xUnsmoothed,yUnsmoothed,timestepsize):
    XSmoothed = xUnsmoothed
    YSmoothed = []
    YSmoothed.append(yUnsmoothed[0])
    if xUnsmoothed[0] + timestepsize == xUnsmoothed[1] and xUnsmoothed[1] + timestepsize == xUnsmoothed[2]:
        YSmoothed.append((yUnsmoothed[0]+yUnsmoothed[1]+yUnsmoothed[2])/3)
    else:
        YSmoothed.append(yUnsmoothed[1])
    for i in range(len(xUnsmoothed)-4):
        if xUnsmoothed[i] + timestepsize == xUnsmoothed[i+1] and xUnsmoothed[i+1] + timestepsize == xUnsmoothed[i+2] and xUnsmoothed[i+2] + timestepsize == xUnsmoothed[i+3] and xUnsmoothed[i+3] + timestepsize == xUnsmoothed[i+4]:
            fivePointAverage = (0.2*(yUnsmoothed[i]+yUnsmoothed[i+1]+yUnsmoothed[i+2]+yUnsmoothed[i+3]+yUnsmoothed[i+4]))
            YSmoothed.append(fivePointAverage)
        elif xUnsmoothed[i+1] + timestepsize == xUnsmoothed[i+2] and xUnsmoothed[i+2] + timestepsize == xUnsmoothed[i+3]:
            threePointAverage = ((yUnsmoothed[i+1]+yUnsmoothed[i+2]+yUnsmoothed[i+3])/3)
            YSmoothed.append(threePointAverage)
        else:
            YSmoothed.append(yUnsmoothed[i+2])
    if xUnsmoothed[-3] + timestepsize == xUnsmoothed[-2] and xUnsmoothed[-2] + timestepsize == xUnsmoothed[-1]:
        YSmoothed.append((yUnsmoothed[-3]+yUnsmoothed[-2]+yUnsmoothed[-1])/3)
    else:
        YSmoothed.append(yUnsmoothed[-2])
    YSmoothed.append(yUnsmoothed[-1])
    return(XSmoothed,YSmoothed)
def freqbinfinder(ffs,freqvalues,timeindicies):
    Frequencies = freqvalues
    Intensities = []
    for i in range(len(freqvalues)):
        for j in range(len(ffs)):
            if ffs[j]==freqvalues[i]:
                #print(Sps[int(NewCorrelatedAspecNormRescaledX[i])][j])
                Intensities.append(Sps[int(timeindicies[i])][j])
    return(Intensities)
def nearestSevenIntegratorPeakDetector(array):
    import numpy as np
    IntegratedArray = []
    IntegratedArray.append(array[0])
    IntegratedArray.append(array[1])
    IntegratedArray.append(array[2])
    for i in range(len(array)-6):
        IAelement = array[i]+array[i+1]+array[i+2]+array[i+3]+array[i+4]+array[i+5]+array[i+6]
        IntegratedArray.append(IAelement)
    
    IntegratedArray.append(array[-3])
    IntegratedArray.append(array[-2])
    IntegratedArray.append(array[-1])
    MaxValue = np.max(IntegratedArray)
    MaxIndex = np.argmax(IntegratedArray)
    
    return(IntegratedArray,MaxValue,MaxIndex)
def Thresholder(Xtobethresholded,Ytobethresholded,Thresholdvalue):
    Xoutput = []
    Youtput = []
    for i in range(len(Xtobethresholded)):
        if Xtobethresholded[i] > Thresholdvalue:
            Xoutput.append(Xtobethresholded[i])
            Youtput.append(Ytobethresholded[i])
    return(Xoutput,Youtput)
def CeilingThresholder(Xtobethresholded,Ytobethresholded,Thresholdvalue):
    Xoutput = []
    Youtput = []
    for i in range(len(Xtobethresholded)):
        if Xtobethresholded[i] <= Thresholdvalue:
            Xoutput.append(Xtobethresholded[i])
            Youtput.append(Ytobethresholded[i])
    return(Xoutput,Youtput)
def angular_separation(lon1, lat1, lon2, lat2):
    sdlon = np.sin(lon2 - lon1)
    cdlon = np.cos(lon2 - lon1)
    slat1 = np.sin(lat1)
    slat2 = np.sin(lat2)
    clat1 = np.cos(lat1)
    clat2 = np.cos(lat2)

    num1 = clat2 * sdlon
    num2 = clat1 * slat2 - slat1 * clat2 * cdlon
    denominator = slat1 * slat2 + clat1 * clat2 * cdlon

    return np.arctan2(np.hypot(num1, num2), denominator)
def nearest23IntegratorPeakDetector(array):
    import numpy as np
    IntegratedArray = []
    IntegratedArray.append(array[0])
    IntegratedArray.append(array[1])
    IntegratedArray.append(array[2])
    
    IntegratedArray.append(array[3])
    IntegratedArray.append(array[4])
    IntegratedArray.append(array[5])
    IntegratedArray.append(array[6])
    IntegratedArray.append(array[7])
    IntegratedArray.append(array[8])
    IntegratedArray.append(array[9])
    IntegratedArray.append(array[10])
    
    for i in range(len(array)-22):
        IAelement = array[i]+array[i+1]+array[i+2]+array[i+3]+array[i+4]+array[i+5]+array[i+6]+array[i+7]+array[i+8]+array[i+9]+array[i+10]+array[i+11]+array[i+12]+array[i+13]+array[i+14]+array[i+15]+array[i+16]+array[i+17]+array[i+18]+array[i+19]+array[i+20]+array[i+21]+array[i+22]
        IntegratedArray.append(IAelement)
    
    IntegratedArray.append(array[-11])
    IntegratedArray.append(array[-10])
    IntegratedArray.append(array[-9])
    IntegratedArray.append(array[-8])
    IntegratedArray.append(array[-7])
    IntegratedArray.append(array[-6])
    IntegratedArray.append(array[-5])
    IntegratedArray.append(array[-4])
    
    IntegratedArray.append(array[-3])
    IntegratedArray.append(array[-2])
    IntegratedArray.append(array[-1])
    MaxValue = np.max(IntegratedArray)
    MaxIndex = np.argmax(IntegratedArray)
    
    return(IntegratedArray,MaxValue,MaxIndex)
def CrossCorrelationCalculation(tsp,ffs,Sps,ASpec,printGraphTrue,saveFigTrue):
    import numpy as np
    IntegratedCorrelation3 = []
    IntegratedCorrelation3time = []
    Correlation3 = []
    Correlation3Maxima = []
    TimeAdditive = (tsp[1]-tsp[0])/2
    
    Correlated5,lags = correlator(Sps[0],np.array(ASpec))
    
    for i in range(len(Sps)):
        print(i)
        SpsGraphToBeCorrelated = Sps[i]
        
        Correlated3,lags = correlator(SpsGraphToBeCorrelated,np.array(ASpec))
        
        differencetemplate = Correlated3 - Correlated5
        n = np.median(Correlated3 - Correlated5)
        differenceCorrelated3 = differencetemplate-n*np.ones(len(Correlated3))
        IntegratedDifferenceCorrelation3,maximum,argmax = nearest23IntegratorPeakDetector(differenceCorrelated3)

        Correlation3Maxima.append(maximum)
        IntegratedCorrelation3.append(argmax)
        IntegratedCorrelation3time.append(i+TimeAdditive)
        Correlation3.append(Correlated3)

        if printGraphTrue == 1:
            plt.figure(figsize=(6.4, 4.8),facecolor = 'aliceblue',dpi=1000)
            plt.plot(ffs, Correlated3, label = 'ASpec as Template')
            plt.plot(ffs, Correlated5, label = 'Sps[0]andAspec as Template')
            plt.title('Correlation %i' %i)
            plt.legend()
        if saveFigTrue == 1:
            plt.savefig('Correlations/Correlation %i' %i)
        if printGraphTrue == 1 or saveFigTrue == 1:
            plt.show()
    HzPerBin = int(ffs[1]-ffs[0])
    IntegratedCorrelation3ScaledToMatchFrequency = scalar(IntegratedCorrelation3,HzPerBin,HzPerBin*(-1*len(ASpec)/2)+ffs[np.argmax(ASpec)])
    return(IntegratedCorrelation3time,IntegratedCorrelation3ScaledToMatchFrequency,Correlation3,Correlation3Maxima)
def ASpecRescaler(Sps,ASpec):
    Spsmed = np.median(Sps)
    Spsmax = np.max(Sps)
    ASpecmed = np.median(ASpec)
    ASpecmax = np.max(ASpec)
    a = (Spsmax-Spsmed)/(ASpecmax-ASpecmed)
    b = Spsmax - a*ASpecmax
    ASpecRescaled = scalar(ASpec,a,b)
    return(ASpecRescaled)
def DopplerShiftCalculator(year, month, day, hour, minute, second,timeintervalseconds,frequencyinit,frequencysubtraction,ffs,line1,line2,Sps,timeoffset):
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    ts = load.timescale()
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    observationlengthinminutes = 30
    c = 299792458 # speed of light in m/s
    angle = 0
    ObjectName = 'TIANHE'
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    print(beginningtimeofobservation)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    #timeoffset = timeoffset
    HzPerBin = int(ffs[1]-ffs[0])
    frequency = frequencyinit*np.ones(observationlengthinminutes*60)
    totaldopplershift = np.ones(observationlengthinminutes*60)
    AngularDistanceStowed = np.ones(len(Sps))
    SatelliteLatitude = np.ones(len(Sps))
    differenceMP = satellite - mountpleasant
    differenceNZ = satellite - newZealand
    differenceKe = satellite - Katherine
    for i in range(observationlengthinminutes*60):
        time1 = ts.utc(year, month, day, hour, minute+0, second+1*i+timeoffset)
        time2 = ts.utc(year, month, day, hour, minute+0, second+1*i+1+timeoffset)
        topocentricMP1 = differenceMP.at(time1)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        topocentricMP2 = differenceMP.at(time2)
        altMP2, azMP2, heightMP2 = topocentricMP2.altaz()
        displacementin1secondinmetres = -(heightMP2.m - heightMP1.m)
    

        if i <= len(Sps)-1:
            lon1 = azMP1.degrees*np.pi/180
            lat1 = altMP1.degrees*np.pi/180
            lon2 = 0*np.pi/180
            lat2 = 88*np.pi/180
            AngularDistanceStowed[i] = angular_separation(lon1, lat1, lon2, lat2)*180/np.pi
            SatelliteLatitude[i] = lat1



        velocity = displacementin1secondinmetres/timeintervalseconds
        fdopplershift = (1*velocity*np.cos(angle)*frequencyinit)/(c - velocity)
        totaldopplershift[i] = fdopplershift/HzPerBin #Hz per bin - Keep an eye on this value
        frequency[i] = frequencyinit + fdopplershift

    h = len(Sps)
    tdslength = totaldopplershift[:h]
    tdslength2 = np.ones(h)
    Rolled=Sps.copy()

    for i in range(len(Sps)):
        Rolled[i] = np.roll(Rolled[i],-int(totaldopplershift[i]),axis=0)
    Nspec = len(Sps)
    Aspec = np.sum(Rolled, axis=0) / Nspec
    
    #print(frequencyinit,frequencysubtraction)
    #print(tdslength)

    ExpectedFrequencies =  [(i * HzPerBin +int(frequencyinit-frequencysubtraction)) for i in tdslength]
    OverlayTimeTrimmed = []
    OverlayFrequencyTrimmed = []
    for i in range(len(Sps)):
        currenttime = ts.utc(year, month, day, hour, minute+0, second+timeintervalseconds*i+timeoffset)
        topocentricMP1 = differenceMP.at(currenttime)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        if altMP1.degrees > 0:
            OverlayTimeTrimmed.append(timeintervalseconds*i+0.5)
            OverlayFrequencyTrimmed.append(ExpectedFrequencies[i])
    
    print("Process Complete")
    ASpecRescaled = ASpecRescaler(Sps,Aspec)
    return(Rolled,OverlayTimeTrimmed,OverlayFrequencyTrimmed,ASpecRescaled)
def DopplerShiftCalculatorKe(year, month, day, hour, minute, second,timeintervalseconds,frequencyinit,frequencysubtraction,ffs,line1,line2,Sps,timeoffset):
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    ts = load.timescale()
    mountpleasant = wgs84.latlon(-42.805, 147.439)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    observationlengthinminutes = 30
    c = 299792458 # speed of light in m/s
    angle = 0
    ObjectName = 'TIANHE'
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    beginningtimeofobservation = ts.utc(year, month, day, hour, minute, second)
    print(beginningtimeofobservation)
    concludedtimeofobservation = ts.utc(year, month, day, hour, minute+observationlengthinminutes, second)
    #timeoffset = timeoffset
    HzPerBin = int(ffs[1]-ffs[0])
    frequency = frequencyinit*np.ones(observationlengthinminutes*60)
    totaldopplershift = np.ones(observationlengthinminutes*60)
    AngularDistanceStowed = np.ones(len(Sps))
    SatelliteLatitude = np.ones(len(Sps))
    differenceMP = satellite - mountpleasant
    differenceNZ = satellite - newZealand
    differenceKe = satellite - Katherine
    for i in range(observationlengthinminutes*60):
        time1 = ts.utc(year, month, day, hour, minute+0, second+1*i+timeoffset)
        time2 = ts.utc(year, month, day, hour, minute+0, second+1*i+1+timeoffset)
        topocentricMP1 = differenceKe.at(time1)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        topocentricMP2 = differenceKe.at(time2)
        altMP2, azMP2, heightMP2 = topocentricMP2.altaz()
        displacementin1secondinmetres = -(heightMP2.m - heightMP1.m)
    

        if i <= len(Sps)-1:
            lon1 = azMP1.degrees*np.pi/180
            lat1 = altMP1.degrees*np.pi/180
            lon2 = 0*np.pi/180
            lat2 = 88*np.pi/180
            AngularDistanceStowed[i] = angular_separation(lon1, lat1, lon2, lat2)*180/np.pi
            SatelliteLatitude[i] = lat1



        velocity = displacementin1secondinmetres/timeintervalseconds
        fdopplershift = (1*velocity*np.cos(angle)*frequencyinit)/(c - velocity)
        totaldopplershift[i] = fdopplershift/HzPerBin #Hz per bin - Keep an eye on this value
        frequency[i] = frequencyinit + fdopplershift

    h = len(Sps)
    tdslength = totaldopplershift[:h]
    tdslength2 = np.ones(h)
    Rolled=Sps.copy()

    for i in range(len(Sps)):
        Rolled[i] = np.roll(Rolled[i],-int(totaldopplershift[i]),axis=0)
    Nspec = len(Sps)
    Aspec = np.sum(Rolled, axis=0) / Nspec

    ExpectedFrequencies =  [(i * HzPerBin +int(frequencyinit-frequencysubtraction)) for i in tdslength]
    OverlayTimeTrimmed = []
    OverlayFrequencyTrimmed = []
    for i in range(len(Sps)):
        currenttime = ts.utc(year, month, day, hour, minute+0, second+timeintervalseconds*i+timeoffset)
        topocentricMP1 = differenceKe.at(currenttime)
        altMP1, azMP1, heightMP1 = topocentricMP1.altaz()
        if altMP1.degrees > 0:
            OverlayTimeTrimmed.append(timeintervalseconds*i+0.5)
            OverlayFrequencyTrimmed.append(ExpectedFrequencies[i])
    
    print("Process Complete")
    ASpecRescaled = ASpecRescaler(Sps,Aspec)
    return(Rolled,OverlayTimeTrimmed,OverlayFrequencyTrimmed,ASpecRescaled,tdslength)
def RMSECalculator(ListOfResiduals):
    import numpy as np
    totalOfSquares = 0
    for i in range(len(ListOfResiduals)):
        totalOfSquares += (ListOfResiduals[i])*(ListOfResiduals[i])
    RMSE = np.sqrt(totalOfSquares/len(ListOfResiduals))
    return(RMSE)
def sensitivityToTLE(Threshold,Sensitivity,Time,Frequency,Intensity,ExpectedTime,ExpectedFrequency,ReturnFullRange):
    CorrelationMaximaWithinSensitivityRangeAboveThreshold = []
    FrequenciesWithinSensitivityRangeAboveThreshold = []
    TimeBinsWithinSensitivityRangeAboveThreshold = []
    
    CorrelationMaximaWithinSensitivityRangeBelowThreshold = []
    FrequenciesWithinSensitivityRangeBelowThreshold = []
    TimeBinsWithinSensitivityRangeBelowThreshold = []
    
    CorrelationMaximaOutsideSensitivityRangeAboveThreshold = []
    FrequenciesOutsideSensitivityRangeAboveThreshold = []
    TimeBinsOutsideSensitivityRangeAboveThreshold = []
    
    CorrelationMaximaOutsideSensitivityRangeBelowThreshold = []
    FrequenciesOutsideSensitivityRangeBelowThreshold = []
    TimeBinsOutsideSensitivityRangeBelowThreshold = []
    
    print(ExpectedTime[0],int(ExpectedTime[0]))
    
    FrequencyTrimmed = Frequency[int(ExpectedTime[0]):int(ExpectedTime[-1]+1)]
    IntensityTrimmed = Intensity[int(ExpectedTime[0]):int(ExpectedTime[-1]+1)]
    
    timePreRise = []
    timePostSet = []
    if ReturnFullRange == 1:
        for i in range(len(Time)):
            if Time[i] < ExpectedTime[0]:
                timePreRise.append(Time[i])
        for i in range(len(timePreRise)):
            if Frequency[i] >= Threshold:
                CorrelationMaximaOutsideSensitivityRangeAboveThreshold.append(Intensity[i])
                FrequenciesOutsideSensitivityRangeAboveThreshold.append(Frequency[i])
                TimeBinsOutsideSensitivityRangeAboveThreshold.append(timePreRise[i])
            elif Frequency[i] < Threshold:    
                CorrelationMaximaOutsideSensitivityRangeBelowThreshold.append(Intensity[i])
                FrequenciesOutsideSensitivityRangeBelowThreshold.append(Frequency[i])
                TimeBinsOutsideSensitivityRangeBelowThreshold.append(timePreRise[i])
        for i in range(len(Time)):
            if Time[i] > ExpectedTime[-1]:
                timePostSet.append(Time[i])
        
    #print("lenFrequencyTrimmed",len(FrequencyTrimmed))
    #print("lenExpectedFrequency",len(ExpectedFrequency))
    DifferencesPredictedvsObservedFrequencies = []
    for i in range(len(ExpectedTime)):
        diff = FrequencyTrimmed[i]-ExpectedFrequency[i]
        DifferencesPredictedvsObservedFrequencies.append(diff)
    #print("FrequencyTrimmed",(FrequencyTrimmed))
    #print("ExpectedFrequency",(ExpectedFrequency))
    ListOfResidualsForBestPoints = []
    ListOfResidualsAboveThreshold = []
    ListOfResidualsWithinSensitivity = [] 
    for i in range(len(ExpectedTime)):
        if DifferencesPredictedvsObservedFrequencies[i] >= -1*Sensitivity and DifferencesPredictedvsObservedFrequencies[i] <= Sensitivity and IntensityTrimmed[i] >= Threshold:
            CorrelationMaximaWithinSensitivityRangeAboveThreshold.append(IntensityTrimmed[i])
            FrequenciesWithinSensitivityRangeAboveThreshold.append(FrequencyTrimmed[i])
            TimeBinsWithinSensitivityRangeAboveThreshold.append(ExpectedTime[i])
            ListOfResidualsForBestPoints.append(DifferencesPredictedvsObservedFrequencies[i])
            ListOfResidualsAboveThreshold.append(DifferencesPredictedvsObservedFrequencies[i])
            ListOfResidualsWithinSensitivity.append(DifferencesPredictedvsObservedFrequencies[i])
        elif DifferencesPredictedvsObservedFrequencies[i] >= -1*Sensitivity and DifferencesPredictedvsObservedFrequencies[i] <= Sensitivity and IntensityTrimmed[i] < Threshold:
            CorrelationMaximaWithinSensitivityRangeBelowThreshold.append(IntensityTrimmed[i])
            FrequenciesWithinSensitivityRangeBelowThreshold.append(FrequencyTrimmed[i])
            TimeBinsWithinSensitivityRangeBelowThreshold.append(ExpectedTime[i])
            ListOfResidualsWithinSensitivity.append(DifferencesPredictedvsObservedFrequencies[i])
        elif IntensityTrimmed[i] >= Threshold:
            CorrelationMaximaOutsideSensitivityRangeAboveThreshold.append(IntensityTrimmed[i])
            FrequenciesOutsideSensitivityRangeAboveThreshold.append(FrequencyTrimmed[i])
            TimeBinsOutsideSensitivityRangeAboveThreshold.append(ExpectedTime[i])
            ListOfResidualsAboveThreshold.append(DifferencesPredictedvsObservedFrequencies[i])
        else:
            CorrelationMaximaOutsideSensitivityRangeBelowThreshold.append(IntensityTrimmed[i])
            FrequenciesOutsideSensitivityRangeBelowThreshold.append(FrequencyTrimmed[i])
            TimeBinsOutsideSensitivityRangeBelowThreshold.append(ExpectedTime[i])
            
    if ReturnFullRange == 1:
        for i in range(len(timePostSet)):
            if Frequency[i+len(ExpectedTime)+len(timePreRise)] >= Threshold:
                CorrelationMaximaOutsideSensitivityRangeAboveThreshold.append(Intensity[i+len(ExpectedTime)+len(timePreRise)])
                FrequenciesOutsideSensitivityRangeAboveThreshold.append(Frequency[i+len(ExpectedTime)+len(timePreRise)])
                TimeBinsOutsideSensitivityRangeAboveThreshold.append(timePostSet[i])
            elif Frequency[i+len(ExpectedTime)+len(timePreRise)] < Threshold:    
                CorrelationMaximaOutsideSensitivityRangeBelowThreshold.append(Intensity[i+len(ExpectedTime)+len(timePreRise)])
                FrequenciesOutsideSensitivityRangeBelowThreshold.append(Frequency[i+len(ExpectedTime)+len(timePreRise)])
                TimeBinsOutsideSensitivityRangeBelowThreshold.append(timePostSet[i])
    (print(ListOfResidualsForBestPoints))
    RMSE1 = RMSECalculator(ListOfResidualsForBestPoints)
    print("RMSE for points within sensitivity and above threshold is",RMSE1,"Hz")
    RMSE2 = RMSECalculator(ListOfResidualsAboveThreshold)
    print("RMSE for points above threshold is",RMSE2,"Hz")
    RMSE3 = RMSECalculator(ListOfResidualsWithinSensitivity)
    print("RMSE for points within sensitivity is",RMSE3,"Hz")
    RMSE4 = RMSECalculator(DifferencesPredictedvsObservedFrequencies)
    print("RMSE for all points is",RMSE4,"Hz")
    print(ListOfResidualsForBestPoints)
    return(CorrelationMaximaWithinSensitivityRangeAboveThreshold,FrequenciesWithinSensitivityRangeAboveThreshold,TimeBinsWithinSensitivityRangeAboveThreshold,CorrelationMaximaWithinSensitivityRangeBelowThreshold,FrequenciesWithinSensitivityRangeBelowThreshold,TimeBinsWithinSensitivityRangeBelowThreshold,CorrelationMaximaOutsideSensitivityRangeAboveThreshold,FrequenciesOutsideSensitivityRangeAboveThreshold,TimeBinsOutsideSensitivityRangeAboveThreshold,CorrelationMaximaOutsideSensitivityRangeBelowThreshold,FrequenciesOutsideSensitivityRangeBelowThreshold,TimeBinsOutsideSensitivityRangeBelowThreshold,DifferencesPredictedvsObservedFrequencies)

def AllSiteVisibility(ObjectName,line1,line2,Hb,Yg,Ke,Cd,Bt,year, month, day, hour, minute, second, numberofdays, LowestVisibleAltitude):

    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import numpy as np
    from datetime import datetime
    from pytz import timezone
    mountpleasant = wgs84.latlon(-42.8055, 147.438)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    BisdeeTier = wgs84.latlon(-42.4311,147.2878)
    ts = load.timescale()

    satellite = EarthSatellite(line1, line2, ObjectName, ts)

    differenceHb = satellite - mountpleasant
    differenceNZ = satellite - newZealand
    differenceYg = satellite - Yarragadee
    differenceKe = satellite - Katherine
    differenceCd = satellite - Ceduna
    differenceBt = satellite - BisdeeTier

    observationtime = ts.utc(year, month, day, hour, minute, second)
    days = observationtime - satellite.epoch
    
    print(satellite)
    print('Time is {:.3f} days after TLE epoch time'.format(days))
    
    NEL0 = 0
    NELALL = LowestVisibleAltitude
    
    #Nominal Elevation Limits (degrees)
    NELHb = NELALL
    NELYg = NELALL
    NELKe = NELALL
    NELCd = NELALL
    NELBt = NELALL

    minutesperhour = 60
    hoursperday = 24
    #Boolean Operators
    UHb = 1
    VHb = 0

    UYg = 1
    VYg = 0

    UKe = 1
    VKe = 0

    UCd = 1
    VCd = 0
    
    UBt = 1
    VBt = 0
    
    observationminutes = int(minutesperhour*hoursperday*numberofdays)

    for i in range(observationminutes):

        #timing section begins
        observationtime = ts.utc(year, month, day, hour, minute+i)
        minutebefore = ts.utc(year, month, day, hour, minute+i-1)
        minuteafter = ts.utc(year, month, day, hour, minute+1+i)

        observationtimeaest = ts.utc(year, month, day, hour+10, minute+i)
        minutebeforeaest = ts.utc(year, month, day, hour+10, minute+i-1)
        minuteafteraest = ts.utc(year, month, day, hour+10, minute+1+i)

        observationtimeaedt = ts.utc(year, month, day, hour+11, minute+i)
        minutebeforeaedt = ts.utc(year, month, day, hour+11, minute+i-1)
        minuteafteraedt = ts.utc(year, month, day, hour+11, minute+1+i)

        odt1 = str(observationtimeaest.utc_strftime())[0:19]
        mbdt = str(minutebeforeaest.utc_strftime())[0:19]
        #timing section ends

        #location section begins
        topocentricHb = differenceHb.at(observationtime)
        altHb, azHb, heightHb = topocentricHb.altaz()
        topocentricNZ = differenceNZ.at(observationtime)
        altNZ, azNZ, heightNZ = topocentricNZ.altaz()
        topocentricYg = differenceYg.at(observationtime)
        altYg, azYg, heightYg = topocentricYg.altaz()
        topocentricKe = differenceKe.at(observationtime)
        altKe, azKe, heightKe = topocentricKe.altaz()
        topocentricCd = differenceCd.at(observationtime)
        altCd, azCd, heightCd = topocentricCd.altaz()
        topocentricBt = differenceBt.at(observationtime)
        altBt, azBt, heightBt = topocentricBt.altaz()
        #location section ends
    ##################################################################################################################################    
        #Hobart Visibility Printer Section Begins
        if altHb.degrees > NELHb and UHb == 1 and Hb == 1:
            print('| UTC date and time:', observationtime.utc_strftime(),'| AEST date and time:', odt1,'AEST','| Visibility Begins Hb |')
            UHb = 0
            VHb = 1
        elif altHb.degrees > NELHb and UHb == 0 and Hb == 1:
            pass     
        else:
            UHb = 1

        if UHb == 1 and VHb == 1 and Hb == 1:
            print('| UTC date and time:', minutebefore.utc_strftime(),'| AEST date and time:', mbdt,'AEST','| Visibility Ending Hb |')
    #        print('|')
            VHb = 0
        #Hobart Visibility Printer Section Ends
    ##################################################################################################################################    
        #Yarragadee Visibility Printer Section Begins
        if altYg.degrees > NELYg and UYg == 1 and Yg == 1:
            print('| UTC date and time:', observationtime.utc_strftime(),'| AEST date and time:', odt1,'AEST','| Visibility Begins Yg |')
            UYg = 0
            VYg = 1
        elif altYg.degrees > NELYg and UYg == 0 and Yg == 1:
            pass     
        else:
            UYg = 1

        if UYg == 1 and VYg == 1 and Yg == 1:
            print('| UTC date and time:', minutebefore.utc_strftime(),'| AEST date and time:', mbdt,'AEST','| Visibility Ending Yg |')
    #        print('|')
            VYg = 0
        #Yarragadee Visibility Printer Section Ends 
    ##################################################################################################################################    
        #Katherine Visibility Printer Section Begins
        if altKe.degrees > NELKe and UKe == 1 and Ke == 1:
            print('| UTC date and time:', observationtime.utc_strftime(),'| AEST date and time:', odt1,'AEST','| Visibility Begins Ke |')
            UKe = 0
            VKe = 1
        elif altKe.degrees > NELKe and UKe == 0 and Ke == 1:
            pass     
        else:
            UKe = 1

        if UKe == 1 and VKe == 1 and Ke == 1:
            print('| UTC date and time:', minutebefore.utc_strftime(),'| AEST date and time:', mbdt,'AEST','| Visibility Ending Ke |')
    #        print('|')
            VKe = 0
        #Katherine Visibility Printer Section Ends 
    ##################################################################################################################################
        #Ceduna Visibility Printer Section Begins
        if altCd.degrees > NELCd and UCd == 1 and Cd == 1:
            print('| UTC date and time:', observationtime.utc_strftime(),'| AEST date and time:', odt1,'AEST','| Visibility Begins Cd |')
            UCd = 0
            VCd = 1
        elif altCd.degrees > NELCd and UCd == 0 and Cd == 1:
            pass     
        else:
            UCd = 1

        if UCd == 1 and VCd == 1 and Cd == 1:
            print('| UTC date and time:', minutebefore.utc_strftime(),'| AEST date and time:', mbdt,'AEST','| Visibility Ending Cd |')
    #        print('|')
            VCd = 0
        #Ceduna Visibility Printer Section Ends 
    ##################################################################################################################################
        #Bisdee Tier Visibility Printer Section Begins
        if altBt.degrees > NELBt and UBt == 1 and Bt == 1:
            print('| UTC date and time:', observationtime.utc_strftime(),'| AEST date and time:', odt1,'AEST','| Visibility Begins Bt |')
            UBt = 0
            VBt = 1
        elif altBt.degrees > NELBt and UBt == 0 and Bt == 1:
            pass     
        else:
            UBt = 1

        if UBt == 1 and VBt == 1 and Bt == 1:
            print('| UTC date and time:', minutebefore.utc_strftime(),'| AEST date and time:', mbdt,'AEST','| Visibility Ending Bt |')
    #        print('|')
            VBt = 0
        #Bisdee Tier Visibility Printer Section Ends 
    ################################################################################################################################## 